# Preprocessing Open Agenda data

>L’objectif est de démontrer aux équipes produit et marketing que notre plateforme peut intégrer un chatbot capable de **répondre aux questions des utilisateurs** en s’appuyant sur les **données d’événements** disponibles via l’API Open Agenda.

- *Zone géographique* : Occitanie
- *Récent (moins de 1 an)* : 10-2025 -> 10-2026

In [18]:
import os
import re
import requests
import json
import pandas as pd
from dotenv import load_dotenv

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from mistralai.client import Mistral

%load_ext autoreload
%autoreload 2

load_dotenv()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


True

In [6]:
client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

In [19]:
# 1. Chargement depuis l'API Opendatasoft
BASE_URL = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/evenements-publics-openagenda"

COLUMNS = [
    # identité / texte
    "uid", "title_fr", "description_fr", "longdescription_fr", "keywords_fr",
    "conditions_fr", "originagenda_title", "canonicalurl",
    # dates
    "daterange_fr", "firstdate_begin", "lastdate_end", "updatedat",
    # lieu
    "location_name", "location_address", "location_postalcode", "location_city",
    "location_department", "location_coordinates", "location_website", "location_phone",
    # infos pratiques
    "age_min", "age_max", "status", "attendancemode", "registration",
    "accessibility_label_fr",
]

params = {
    "refine": "location_region:Occitanie",
    # événements qui chevauchent la fenêtre oct. 2025 -> oct. 2026
    "where": "lastdate_end >= date'2025-01-01' AND firstdate_begin <= date'2026-10-29'",
    "select": ",".join(COLUMNS),
    # "limit": 2000,  # pour tester sur un échantillon
}

resp = requests.get(f"{BASE_URL}/exports/json", params=params, timeout=300)
resp.raise_for_status()
df = pd.DataFrame(resp.json())


In [20]:
# 2. Décodage des colonnes JSON texte
def is_empty(value):
    """True pour None, NaN, pd.NA, chaîne vide ou liste vide."""
    if isinstance(value, (list, dict)):
        return len(value) == 0
    return value is None or pd.isna(value) or value == ""

def label_fr(value):
    """Extrait le libellé français d'un champ JSON {id, label: str | {fr: ...}}."""
    if is_empty(value):
        return None
    label = json.loads(value).get("label")
    if isinstance(label, dict):
        return label.get("fr")
    return label

def registration_links(value):
    """Retourne la liste des liens/contacts d'inscription."""
    if is_empty(value):
        return []
    return [str(r["value"]) for r in json.loads(value) if r.get("value")]

df["status"] = df["status"].map(label_fr)                  # "Programmé", "Annulé", ...
df["attendancemode"] = df["attendancemode"].map(label_fr)  # "Sur place", "En ligne", "Mixte"
df["registration"] = df["registration"].map(registration_links)


In [21]:
# 3. Nettoyage
df = df[~df["status"].isin(["Annulé", "Reporté"])].drop_duplicates(subset="uid")

def clean_html(text):
    """Retire les balises HTML et normalise les espaces."""
    if is_empty(text):
        return ""
    text = re.sub(r"<[^>]+>", " ", str(text))
    return re.sub(r"\s+", " ", text).strip()


In [22]:
# 4. Construction des documents LangChain
def build_page_content(row):
    """Texte vectorisé : tout ce sur quoi l'utilisateur peut poser une question."""
    lieu = ", ".join(
        str(row[c]) for c in ["location_name", "location_address", "location_city", "location_department"]
        if not is_empty(row[c])
    )
    fields = {
        "Titre": row["title_fr"],
        "Dates": row["daterange_fr"],
        "Lieu": lieu,
        "Organisateur": row["originagenda_title"],
        "Résumé": clean_html(row["description_fr"]),
        "Description": clean_html(row["longdescription_fr"]),
        "Mots-clés": ", ".join(row["keywords_fr"]) if isinstance(row["keywords_fr"], list) else None,
        "Conditions": row["conditions_fr"],
        "Âge": f"de {int(row['age_min'])} à {int(row['age_max'])} ans"
               if not is_empty(row["age_min"]) and not is_empty(row["age_max"]) else None,
        "Mode": row["attendancemode"],
        "Statut": row["status"],
        "Accessibilité": row["accessibility_label_fr"],
        "Réservation": ", ".join(row["registration"]) or None,
        "Lien": row["canonicalurl"],
    }
    return "\n".join(f"{k} : {v}" for k, v in fields.items() if not is_empty(v))

def none_if_empty(value):
    return None if is_empty(value) else value

documents = [
    Document(
        page_content=build_page_content(row),
        metadata={
            "uid": row["uid"],
            "title": none_if_empty(row["title_fr"]),
            "city": none_if_empty(row["location_city"]),
            "department": none_if_empty(row["location_department"]),
            "start": none_if_empty(row["firstdate_begin"]),
            "end": none_if_empty(row["lastdate_end"]),
            "url": none_if_empty(row["canonicalurl"]),
        },
    )
    for _, row in df.iterrows()
]

print(f"{len(df)} événements -> {len(documents)} documents")
print(documents[0].page_content)


41665 événements -> 41665 documents
Titre : Tatouages et Prisons. Des murs sur la peau
Dates : 2 avril - 31 août
Lieu : Le Castelet, 18 bis Grande Rue Saint-Michel 31400  Toulouse, France, Toulouse, Haute-Garonne
Organisateur : Monuments de Toulouse
Résumé : Mêlant approche anthropologique, historique, géographique et artistique, l'exposition explore les différents usages et fonctions du tatouage.
Description : Un voyage entre art, histoire et corps marqué Le Castelet de Toulouse ouvre ses portes à une exposition inédite et poignante sur l’univers du tatouage carcéral, entre punition, revendication, et expression artistique. Entre les murs de cette ancienne prison, cette exposition retrace le destin singulier du tatouage, de marque infamante à œuvre d’art. Une plongée dans l’histoire des tatouages en prison, des galères françaises à la mafia japonaise, du Goulag soviétique à l’univers punk, pour comprendre comment une pratique marginalisée est devenue un symbole culturel et artistique 

In [24]:
df.head()

,uid,title_fr,description_fr,longdescription_fr,keywords_fr,conditions_fr,originagenda_title,canonicalurl,daterange_fr,firstdate_begin,...,location_department,location_coordinates,location_website,location_phone,age_min,age_max,status,attendancemode,registration,accessibility_label_fr
0,44571464,Tatouages et Prisons. Des murs sur la peau,"Mêlant approche anthropologique, historique, g...","<h2>Un voyage entre art, histoire et corps mar...","[tatouages et prison, des murs sur la peau, ta...",Entrée gratuite | Visite guidée à 3€,Monuments de Toulouse,https://openagenda.com/monuments-toulouse/even...,2 avril - 31 août,2025-04-02T09:00:00+00:00,...,Haute-Garonne,"{'lon': 1.447495, 'lat': 43.586136}",NaN,NaN,NaN,NaN,Programmé,Sur place,[https://billetterie.monuments.toulouse.fr/sel...,None
1,5122201,"Visite Mon Précieux... ""Une robe en soie à l'a...",Visite autour d'une oeuvre d'art du musée,"<p>En 30 minutes, venez découvrir avec une méd...","[visite, tout public, visite toulouse, mode à ...","Tout public à partir de 11 ans, visites à rese...",Musée des arts précieux Paul-Dupuy,https://openagenda.com/planlibre/events/visite...,"Mardi 31 mars, 16h00",2026-03-31T14:00:00+00:00,...,Haute-Garonne,"{'lon': 1.446869, 'lat': 43.596898}",https://museepauldupuy.toulouse.fr/,05 31 22 95 40,11.0,99.0,Programmé,Sur place,[https://billetterie.musee-dupuy.toulouse.fr/s...,None
2,24377210,"Rencontre Ovins Bio – Mardi 9 décembre, 14h-17...",La Chambre d’agriculture du Lot et Bio 46 vous...,<p>Au programme :<br>📊 Point sur la filière et...,"[ovins, bio, agneaux, cahors]",Sur inscription,Chambre d'agriculture du Lot,https://openagenda.com/toutes-les-chambres-agr...,"Mardi 9 décembre 2025, 14h00",2025-12-09T13:00:00+00:00,...,Lot,"{'lon': 1.434308, 'lat': 44.44898}",NaN,NaN,NaN,NaN,Programmé,Sur place,[https://forms.gle/9fG7utVJwMbSL5nh7],None
3,4154446,"Visite guidée ""Les peintres caravagesques en LSF""",Visite guidée thématique des collections en fr...,<p>Suivez les guides-conférencières du musée p...,None,NaN,Musée des Augustins,https://openagenda.com/musee-des-augustins-tou...,"Dimanche 21 juin, 14h30",2026-06-21T12:30:00+00:00,...,Haute-Garonne,"{'lon': 1.446343, 'lat': 43.600942}",https://augustins.toulouse.fr/,NaN,NaN,NaN,Programmé,Sur place,[https://billetterie.musee-augustins.toulouse....,None
4,26684060,Préparez-Vous pour un Job Dating Transport en ...,Nous vous invitons à participer à notre réunio...,<p>Nous vous invitons à participer à notre réu...,[Se préparer],NaN,Mes événements France Travail,https://openagenda.com/francetravail/events/pr...,"Lundi 21 septembre, 15h30",2026-09-21T13:30:00+00:00,...,Pyrénées-Orientales,"{'lon': 2.871498, 'lat': 42.690102}",NaN,NaN,NaN,NaN,Programmé,Sur place,[https://mesevenementsemploi.francetravail.fr/...,None


In [27]:
df['firstdate_begin']

0        2025-04-02T09:00:00+00:00
1        2026-03-31T14:00:00+00:00
2        2025-12-09T13:00:00+00:00
3        2026-06-21T12:30:00+00:00
4        2026-09-21T13:30:00+00:00
                   ...            
41835    2026-05-21T18:00:00+00:00
41836    2026-05-30T16:30:00+00:00
41837    2026-06-04T18:00:00+00:00
41838    2026-07-15T20:06:17+00:00
41839    2026-09-20T16:00:00+00:00
Name: firstdate_begin, Length: 41665, dtype: str

In [23]:
df["daterange_fr"]

0                   2 avril - 31 août
1                Mardi 31 mars, 16h00
2        Mardi 9 décembre 2025, 14h00
3             Dimanche 21 juin, 14h30
4           Lundi 21 septembre, 15h30
                     ...             
41835             Jeudi 21 mai, 20h00
41836            Samedi 30 mai, 18h30
41837                     4 et 5 juin
41838                 15 - 28 juillet
41839    Dimanche 20 septembre, 18h00
Name: daterange_fr, Length: 41665, dtype: str